In [1]:

import os
import json
import re
import datetime

import config

from time import time, sleep
from uuid import uuid4

from openai import OpenAI
from rdflib import Graph

# Imports propios
from searchInGraph import (
    buscar_frecuentes_por_opcion,
    inferir_valor_adecuado
)

from formatHelper import (
    extraer_support_category,
    extraer_cliente,
    formatear_para_llm,
    arreglar_lista_llm,
    #merge_listas_or,
    #limpiar_lista,
    aplicar_reglas
)



In [2]:
# ============================================
# CELDA 2 - CARGA DEL GRAFO RDF
# ============================================

graph = Graph()

graph.parse(
    config.TTL_FILE_PATH,
    format=config.TTL_FORMAT
)

print("Grafo cargado correctamente")
print(f"Número de triples: {len(graph)}")

Grafo cargado correctamente
Número de triples: 7425543


In [13]:
def obtener_100_clientes_aleatorios(g, prefix_uri="http://repcon.org/schema#"):
    """
    Consulta el grafo para extraer hasta 100 valores distintos y aleatorios 
    del predicado 'int_hasCustomer'.
    """
    
    # Construimos la URI exacta del predicado
    uri_predicado_objetivo = f"<{prefix_uri}int_hasCustomer>"

    # Consulta SPARQL: 
    # - DISTINCT para no repetir el mismo cliente.
    # - ORDER BY RAND() para que cada ejecución devuelva un set diferente.
    # - LIMIT 100 para acotar el resultado.
    query = f"""
    SELECT DISTINCT ?customer
    WHERE {{
        ?incident {uri_predicado_objetivo} ?customer .
    }}
    ORDER BY RAND()
    LIMIT 100
    """

    # Ejecutamos la consulta
    resultados = g.query(query)

    clientes_aleatorios = []
    for row in resultados:
        val_uri = str(row.customer)
        
        # Extraer ID limpio (shorten) usando exactamente tu lógica
        if "#" in val_uri:
            id_limpio = val_uri.split("#")[-1]
        elif "/" in val_uri:
            id_limpio = val_uri.rsplit("/", 1)[-1]
        else:
            id_limpio = val_uri

        clientes_aleatorios.append(id_limpio)

    return clientes_aleatorios

In [14]:
clientes = obtener_100_clientes_aleatorios(graph)
print(clientes)

['company__SOVUI8NGE', 'company__QBNA20T7V', 'ss', 'company__1GR6455ID', 'company__P8EM3ZO35', 'company__QCTRKWQRI', 'company__SAO2BI1Q3', 'company__F7UMNAXNO', 'company_14976568571762302706', 'company__HVY8FPJ74', 'company_14976254351762302678', 'company__3S8A2Y7FV', 'company__PJ42MKUE7', 'company__45TNZMKG4', 'company__IK3K7WJM7', 'company_149763272551762303018', 'company__UQHIM9QXH', 'company__9G1G3MV0P', 'company__VXO7NV5L6', 'company__2AS5C65KH', 'company__HP6K1FYZ8', 'company__7POLMU91Q', 'company__2ZFMBC970', 'company__EZ80EKVHB', 'company__CFD5UKZBE', 'company__10QMPVMGA', 'company__0GQ4QH8N2', 'company__U4S8LL2ZS', 'company_149763036781762302990', 'company__SEI95B7UO', 'company__W0S6TBURD', 'company__V2R70HP4K', 'company_149766626881762303466', 'company_149762888531762302972', 'company__Q9ZGFR6OC', 'company__KWWL01UST', 'company__DQJKY6U6E', 'company_149764242351762303146', 'company_149763138681762303002', 'company__AQZ0PRRZI', 'company__VN9YQ38N3', 'company__EQ7GNRFNB28', 'co

In [ ]:
def obtener_100_categorias_soporte_aleatorias(g, prefix_uri="http://repcon.org/schema#"):
    """
    Consulta el grafo para extraer hasta 100 valores distintos y aleatorios 
    del predicado 'hasSupportCategory'.
    """
    
    # Construimos la URI exacta del predicado
    uri_predicado_objetivo = f"<{prefix_uri}hasSupportCategory>"

    # Consulta SPARQL: 
    # - DISTINCT para no repetir la misma categoría.
    # - ORDER BY RAND() para que cada ejecución devuelva un set diferente.
    # - LIMIT 100 para acotar el resultado.
    query = f"""
    SELECT DISTINCT ?category
    WHERE {{
        ?incident {uri_predicado_objetivo} ?category .
    }}
    ORDER BY RAND()
    LIMIT 100
    """

    # Ejecutamos la consulta
    resultados = g.query(query)

    categorias_aleatorias = []
    for row in resultados:
        val_uri = str(row.category)
        
        # Extraer ID limpio (shorten) usando tu lógica habitual
        if "#" in val_uri:
            id_limpio = val_uri.split("#")[-1]
        elif "/" in val_uri:
            id_limpio = val_uri.rsplit("/", 1)[-1]
        else:
            id_limpio = val_uri

        categorias_aleatorias.append(id_limpio)

    return categorias_aleatorias

In [6]:
suport_categorys = obtener_100_categorias_soporte_aleatorias(graph)
print(suport_categorys)

['supportCategory_149766041762302662', 'supportCategory_149764151762302662', 'supportCategory_149766571762302662', 'supportCategory_149761511762302662', 'supportCategory_1497641831762302663', 'supportCategory_14976212811762302676', 'supportCategory_14976353291762302684', 'supportCategory_149761591762302662', 'supportCategory_1497625421762302663', 'supportCategory_149762111762302662', 'supportCategory_1497647861762302664', 'supportCategory_1497618541762302663', 'supportCategory_149769391762302662', 'supportCategory_149763731762302662', 'supportCategory_149764641762302662', 'supportCategory_149761451762302662', 'supportCategory_1497610291762302662', 'supportCategory_1497668321762302664', 'supportCategory_14976161762302662', 'supportCategory_149764024541762303119', 'supportCategory_149768401762302662', 'supportCategory_1497675931762302665', 'supportCategory_1497661351762302664', 'supportCategory_149767321762302662', 'supportCategory_149761111762302662', 'supportCategory_149763527461762303

In [7]:
import random

def generar_combinaciones_query(lista_categorias, lista_empresas, n):
    """
    Genera 'n' frases aleatorias combinando elementos de dos listas.
    Permite la repetición de categorías de soporte y empresas.
    """
    # Verificamos que las listas no estén vacías para evitar errores
    if not lista_categorias or not lista_empresas:
        print("Error: Una de las listas está vacía.")
        return []

    combinaciones = []
    
    for _ in range(n):
        # Elegimos un elemento aleatorio de cada lista
        categoria_aleatoria = random.choice(lista_categorias)
        empresa_aleatoria = random.choice(lista_empresas)
        
        # Construimos la frase con los valores seleccionados
        frase = f"Hola quiero completar una query. Tengo el {categoria_aleatoria} y la empresa {empresa_aleatoria}"
        
        combinaciones.append(frase)
        
    return combinaciones

In [8]:
#suport_categorys = ["supportCategory_14976110321762302666"]

frases_generadas = generar_combinaciones_query(suport_categorys, clientes, 500)

#extras = ['Hola quiero completar una query. Tengo el supportCategory_14976110321762302666 y la empresa company__W0S6TBURD', 'Hola quiero completar una query. Tengo el supportCategory_149765491762302662 y la empresa company__077OCQVXM', 'Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__17Q32M10L', 'Hola quiero completar una query. Tengo el supportCategory_149765458501762303308 y la empresa company__UQ0EGOKMC']
#
#
#for frase in extras:
#    frases_generadas.insert(frase, 0)
#    
for frase in frases_generadas:
    print(frase)

Hola quiero completar una query. Tengo el supportCategory_149766831762302662 y la empresa company__LNTKSH8FL
Hola quiero completar una query. Tengo el supportCategory_1497611671762302662 y la empresa company__8VD90FMG4
Hola quiero completar una query. Tengo el supportCategory_149762161762302662 y la empresa company__F7UMNAXNO
Hola quiero completar una query. Tengo el supportCategory_1497618541762302663 y la empresa company_149763071541762302992
Hola quiero completar una query. Tengo el supportCategory_14976212811762302676 y la empresa company__LUZC0VLT3
Hola quiero completar una query. Tengo el supportCategory_149765961762302662 y la empresa company__WDY3VEXSFQ254
Hola quiero completar una query. Tengo el supportCategory_149762663151762302943 y la empresa company_149765991071762303375
Hola quiero completar una query. Tengo el supportCategory_149762111762302662 y la empresa company__HVY8FPJ74
Hola quiero completar una query. Tengo el supportCategory_149763021762302662 y la empresa compa

In [4]:
contadores = {"vecesdf": 0, "vecesnv": 0}

def completar_siguiente_campo(mis_datos, flags):



    # ========================================
    # EXTRACCIÓN DE DATOS INICIALES (0 y 1)
    # ========================================
    regla_aplicada = -1

    # ========================================
    # VERIFICAR SI YA ESTÁ COMPLETO
    # ========================================
    if None not in mis_datos and 'None' not in mis_datos:
        #print(f'\nGraphRAG: query acabada. La query es {mis_datos}')
        
        return False, mis_datos, flags, regla_aplicada

    # ========================================
    # BUSCAR CATEGORÍA FALTANTE (SI CORRESPONDE)
    # ========================================
    try:
        cat_buscar = mis_datos.index(None)
    except ValueError:
        cat_buscar = mis_datos.index('None')
    graph_data = buscar_frecuentes_por_opcion(graph, mis_datos, cat_buscar)
    if not graph_data:
        graph_data = inferir_valor_adecuado(graph, mis_datos, cat_buscar)

    print(graph_data)
    
    graph_data, regla_aplicada = aplicar_reglas(
                './textos/reglas_incidentes.json',
                mis_datos, 
                cat_buscar, 
                graph_data, flags
            )

    
    if graph_data:
        mi_opcion = graph_data[0]

    mis_datos[cat_buscar] = mi_opcion
    
    return True, mis_datos, flags, regla_aplicada



In [3]:
contadores = {"vecesdf": 0, "vecesnv": 0}


def procesar_query(texto):
    flags = {"vecesdf": 0, "vecesnv": 0}
    mis_datos = [None, None, None, None, None, None]
    ambas = False
    
    mis_datos[0] = extraer_cliente(texto)

    print(mis_datos)
    mis_datos[1] = extraer_support_category(texto)
    print(mis_datos)
    regla_de_verdad = -1
    
    while True:
        continuar, mis_datos, flags, regla_aplicada = completar_siguiente_campo(mis_datos,flags)

        if regla_aplicada != -1:
            regla_de_verdad = regla_aplicada
        #print(mis_datos)
        if not continuar:

            if flags["vecesdf"] == 2 and flags["vecesnv"] == 2:
                print("Ambas")
                contadores["vecesdf"] +=1
                contadores["vecesnv"] +=1
                ambas = True
                
            else:
                if flags["vecesdf"] >0:
                    contadores["vecesdf"] +=1
    
                if flags["vecesnv"] >0:
                    contadores["vecesnv"] +=1
            
            break
    
    print(f'\nGraphRAG: query acabada. La query es {mis_datos}')

    return mis_datos, ambas, regla_de_verdad

In [5]:
#import concurrent.futures
#def procesar_query(texto):
#    flags = {"vecesdf": 0, "vecesnv": 0}
#    mis_datos = [None, None, None, None, None, None]
#    ambas = False
#    
#    # Variables para indicar al hilo principal qué ocurrió
#    sumo_df = False
#    sumo_nv = False
#    
#    mis_datos[0] = extraer_cliente(texto)
#    mis_datos[1] = extraer_support_category(texto)
#
#    while True:
#        continuar, mis_datos, flags = completar_siguiente_campo(mis_datos, flags)
#        
#        if not continuar:
#            if flags["vecesdf"] == 2 and flags["vecesnv"] == 2:
#                # print("Ambas") # Opcional: comentar prints en paralelo para evitar spam en consola
#                sumo_df = True
#                sumo_nv = True
#                ambas = True
#            else:
#                if flags["vecesdf"] > 0:
#                    sumo_df = True
#                if flags["vecesnv"] > 0:
#                    sumo_nv = True
#            break
#    
#    # Devolvemos el texto original junto con los resultados para no perder la referencia
#    return texto, mis_datos, ambas, sumo_df, sumo_nv

In [5]:
import json

def generar_caso(query, numregla, resultado, ruta_fichero):
    """
    Lee una regla específica de un fichero JSON y genera un nuevo JSON 
    combinando la información proporcionada con los detalles de la regla.
    Si el tipo de regla es 'nv', se guardará como 'noValid'.
    """
    # 1. Cargar el JSON de reglas
    try:
        with open(ruta_fichero, 'r', encoding='utf-8') as f:
            reglas = json.load(f)
    except FileNotFoundError:
        print(f"Error: No se encontró el fichero '{ruta_fichero}'.")
        return None
    except json.JSONDecodeError:
        print("Error: El fichero no contiene un JSON válido.")
        return None

    # 2. Buscar la regla (Asumimos que numregla empieza en 1)
    indice_regla = int(numregla) - 1
    
    if indice_regla < 0 or indice_regla >= len(reglas):
        print(f"Error: La regla número {numregla} no existe en el fichero.")
        return None

    regla_objetivo = reglas[indice_regla]

    # 3. Extraer los valores de la regla
    rule_type = regla_objetivo.get("ruleType", "")
    
    # === MODIFICACIÓN PARA MAPEAR 'nv' A 'noValid' ===
    if rule_type.lower() == "nv":
        rule_type = "noValid"
    
    # Extraemos el primer par clave-valor del objeto "then"
    then_dict = regla_objetivo.get("then", {})
    rule_field = ""
    rule_value = ""
    
    if then_dict:
        # Cogemos la primera clave y su valor asociado
        rule_field = list(then_dict.keys())[0]
        rule_value = list(then_dict.values())[0]

    # 4. Construir la estructura final
    caso = {
        "query": query,
        "expected": resultado,
        "rule": str(numregla),
        "ruletype": rule_type,
        "ruleField": rule_field,
        "ruleValue": rule_value
    }

    # Devolvemos la lista con el diccionario dentro, tal como indica tu formato
    return [caso]

In [22]:
mi_frase =  "Hola quiero completar una query. Tengo el supportCategory_149763371762302662  y la empresa company__UQ0EGOKMC"



# "Hola quiero completar una query. Tengo el supportCategory_1497647941762302664 y la empresa company__LNTKSH8FL" 366

# 'Hola quiero completar una query. Tengo el supportCategory_149762445241762302917 y la empresa company__UQ0EGOKMC'

# "Hola quiero completar una query. Tengo el supportCategory_149763471762302662 y la empresa company__LNTKSH8FL" 239



mi_resultado, b, regla_aplicada = procesar_query(mi_frase)


json_generado = generar_caso(mi_frase, regla_aplicada, mi_resultado, './textos/reglas_incidentes.json')

if json_generado:
        # Imprimir por consola con formato JSON bonito
    print(json.dumps(json_generado, indent=4, ensure_ascii=False))
        
        # Opcional: Guardarlo en un nuevo archivo
        # with open("casos_generados.json", "w", encoding="utf-8") as out_f:
        #     json.dump(json_generado, out_f, indent=4, ensure_ascii=False)


['company__UQ0EGOKMC', None, None, None, None, None]
['company__UQ0EGOKMC', 'supportCategory_149766201291762303400', None, None, None, None]
['typeIncident__1', 'typeIncident__2']

[Regla Aplicada - noValid] (Regla #34) 'typeIncident__1' no es válido para 'hasTypeInc'.
[Regla Aplicada] (Regla #34) Descartando opción principal. Saltando a la siguiente opción.
['incidentOrigin__2', 'incidentOrigin__3', 'incidentOrigin__1', 'incidentOrigin__4']
['supportGroup_14976631762302662', 'supportGroup_149761521762302662', 'supportGroup_149762611762302662', 'supportGroup_14976691762302662']
['employee__266', 'employee__294', 'employee__108', 'employee__239']

GraphRAG: query acabada. La query es ['company__UQ0EGOKMC', 'supportCategory_149766201291762303400', 'typeIncident__2', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__266']
[
    {
        "query": "Hola quiero completar una query. Tengo el supportCategory_149766201291762303400 y la empresa company__UQ0EGOKMC",
        "expe

In [18]:
import json

# Tu nuevo array de entrada
base_array =  ['company__QBNA20T7V',
               'supportCategory_149763484751762303042', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__366']
# Definimos la configuración de los campos con la cantidad de elementos que deben tomar de la lista
configuracion_pasos = [
    {"field": "hasTypeInc", "elementos": 3},
    {"field": "incident_hasOrigin", "elementos": 4},
    {"field": "hasSupportGroup", "elementos": 5},
    {"field": "hasTechnician", "elementos": 6}
]

def get_by_prefix(arr, prefix):
    for item in arr:
        if item.startswith(prefix):
            return item
    return ""

# Extraemos la compañía y la categoría para la query de forma dinámica
company_base = get_by_prefix(base_array, 'company__')
support_category_base = get_by_prefix(base_array, 'supportCategory_')

def generar_casos_progresivos():
    casos = []
    
    for paso in configuracion_pasos:
        query = f"Hola quiero completar una query. Tengo el {support_category_base} y la empresa {company_base}"
        
        # Slicing dinámico: toma desde el inicio hasta el número de elementos requeridos
        expected_acumulado = base_array[:paso["elementos"]]
        
        caso = {
            "query": query,
            "expected": expected_acumulado,
            "rule": "-1",
            "ruletype": "none",
            "ruleField": paso["field"],
            "ruleValue": "none"
        }
        casos.append(caso)
        
    return casos

# Generar y mostrar el JSON estructurado
resultado = generar_casos_progresivos()
print(json.dumps(resultado, indent=4, ensure_ascii=False))

[
    {
        "query": "Hola quiero completar una query. Tengo el supportCategory_149763484751762303042 y la empresa company__QBNA20T7V",
        "expected": [
            "company__QBNA20T7V",
            "supportCategory_149763484751762303042",
            "typeIncident__1"
        ],
        "rule": "-1",
        "ruletype": "none",
        "ruleField": "hasTypeInc",
        "ruleValue": "none"
    },
    {
        "query": "Hola quiero completar una query. Tengo el supportCategory_149763484751762303042 y la empresa company__QBNA20T7V",
        "expected": [
            "company__QBNA20T7V",
            "supportCategory_149763484751762303042",
            "typeIncident__1",
            "incidentOrigin__2"
        ],
        "rule": "-1",
        "ruletype": "none",
        "ruleField": "incident_hasOrigin",
        "ruleValue": "none"
    },
    {
        "query": "Hola quiero completar una query. Tengo el supportCategory_149763484751762303042 y la empresa company__QBNA20T7V",
   

[
    {
        "query": "Hola quiero completar una query. Tengo el supportCategory_149763671762302662 y la empresa company_149762002231762302862",
        "expected": [
            "company_149762002231762302862",
            "supportCategory_149763671762302662",
            "typeIncident__2",
            "incidentOrigin__3",
            "supportGroup_149762881762302662",
            "employee__294"
        ],
        "rule": "8",
        "ruletype": "df",
        "ruleField": "hasSupportGroup",
        "ruleValue": "supportGroup_149762881762302662"
    }
]


In [ ]:
print(contadores)

In [ ]:
casos_df = []
casos_nv = []
casos_ambos = []
casos_dos = []
contadores = {"vecesdf": 0, "vecesnv": 0}
nada = []

#frases_generadas = ['Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__17Q32M10L']

for frase in frases_generadas:
    df_anterior = contadores["vecesdf"]
    nv_anterior = contadores["vecesnv"]
    #print(frase)
    query, ambas = procesar_query(frase)
    
    if ambas:
        #print(f"Loquisimo, se hicieron 2 a la vez: {frase}")
        if len(casos_ambos) <10:
            casos_ambos.insert(0, frase)
    elif (contadores["vecesdf"] == df_anterior + 1) and (contadores["vecesnv"] == nv_anterior + 1):
        #print(f"¡Match! Ambos sumaron 1 en: {frase}")
        if len(casos_dos) <10:
            casos_dos.insert(0, frase)

    elif (contadores["vecesnv"] == nv_anterior + 1): 
        #print(f"¡Match!: {frase}")
        if len(casos_nv) <10:
            casos_nv.insert(0, frase)
    
    elif (contadores["vecesdf"] == df_anterior + 1): 
        #print(f"¡Match!: {frase}")
        if len(casos_df) <10:
            casos_df.insert(0, frase)


    else:
        if len(nada) <10:
            nada.insert(0, frase)

    if len(casos_df) == 10 and len(casos_nv) == 10 and len(casos_dos) == 10 and len(casos_ambos) == 10:
        break

    
    

In [ ]:
print(casos_df)
print(casos_nv)
print(casos_ambos)
print(casos_dos)

In [26]:
#frases_generadas = ['Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__17Q32M10L']

# frases_generadas = ['Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__17Q32M10L', ...]
import io
from contextlib import redirect_stdout

frases_generadas1 = ['Hola quiero completar una query. Tengo el supportCategory_1497617941762302663 y la empresa company_149762002231762302862', 'Hola quiero completar una query. Tengo el supportCategory_149763527461762303053 y la empresa company__QWY4YPRG7', 'Hola quiero completar una query. Tengo el supportCategory_149762916841762302974 y la empresa company__UQHIM9QXH', 'Hola quiero completar una query. Tengo el supportCategory_1497611841762302662 y la empresa company__1GR6455ID', 'Hola quiero completar una query. Tengo el supportCategory_149761991762302662 y la empresa company__2ZFMBC970', 'Hola quiero completar una query. Tengo el supportCategory_149767291231762303563 y la empresa ss', 'Hola quiero completar una query. Tengo el supportCategory_1497681762302662 y la empresa company_149762002231762302862', 'Hola quiero completar una query. Tengo el supportCategory_1497614541762302662 y la empresa ss', 'Hola quiero completar una query. Tengo el supportCategory_149763671762302662 y la empresa company_149767814271762303637', 'Hola quiero completar una query. Tengo el supportCategory_149764151762302662 y la empresa company__CFD5UKZBE']
frases_generadas2 = ['Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__D52NKD9SS', 'Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__17Q32M10L', 'Hola quiero completar una query. Tengo el supportCategory_149763671762302662 y la empresa company__17Q32M10L', 'Hola quiero completar una query. Tengo el supportCategory_1497691561762302665 y la empresa company__17Q32M10L', 'Hola quiero completar una query. Tengo el supportCategory_149765491762302662 y la empresa company__077OCQVXM', 'Hola quiero completar una query. Tengo el supportCategory_149765458501762303308 y la empresa company__UQ0EGOKMC', 'Hola quiero completar una query. Tengo el supportCategory_1497611981762302662 y la empresa company__DQJKY6U6E', 'Hola quiero completar una query. Tengo el supportCategory_149765491762302662 y la empresa company__10QMPVMGA', 'Hola quiero completar una query. Tengo el supportCategory_149763371762302662 y la empresa company__17Q32M10L', 'Hola quiero completar una query. Tengo el supportCategory_149765491762302662 y la empresa company__HVY8FPJ74']
frases_generadas3 = ['Hola quiero completar una query. Tengo el supportCategory_149767060481762303533 y la empresa company_149767070781762303534', 'Hola quiero completar una query. Tengo el supportCategory_149767060481762303533 y la empresa company_149761171471762302766', 'Hola quiero completar una query. Tengo el supportCategory_149762916841762302974 y la empresa company_14976568571762302706', 'Hola quiero completar una query. Tengo el supportCategory_1497617231762302663 y la empresa company__3S8A2Y7FV', 'Hola quiero completar una query. Tengo el supportCategory_149762916841762302974 y la empresa company_149763729881762303079', 'Hola quiero completar una query. Tengo el supportCategory_149767601762302662 y la empresa company_149766665321762303469', 'Hola quiero completar una query. Tengo el supportCategory_14976411762302662 y la empresa company_149767814271762303637', 'Hola quiero completar una query. Tengo el supportCategory_149763471762302662 y la empresa company_149764841521762303225', 'Hola quiero completar una query. Tengo el supportCategory_149763834511762303087 y la empresa company_149764242351762303146', 'Hola quiero completar una query. Tengo el supportCategory_1497631491762302663 y la empresa company__3S8A2Y7FV']
frases_generadas4 = ['Hola quiero completar una query. Tengo el supportCategory_14976110321762302666 y la empresa company__W0S6TBURD', 'Hola quiero completar una query. Tengo el supportCategory_1497610291762302662 y la empresa company__UQ0EGOKMC']
frases_generadas5 = ['Hola quiero completar una query. Tengo el supportCategory_149769471762302662 y la empresa company__IJVARZ08A', 'Hola quiero completar una query. Tengo el supportCategory_1497623551762302663 y la empresa company__Y1XHYEPUS', 'Hola quiero completar una query. Tengo el supportCategory_1497622131762302663 y la empresa company_149766632941762303467', 'Hola quiero completar una query. Tengo el supportCategory_149765639211762303330 y la empresa company__GBKQFD0VA', 'Hola quiero completar una query. Tengo el supportCategory_1497643821762302663 y la empresa company__G93IZU6VM', 'Hola quiero completar una query. Tengo el supportCategory_14976110321762302666 y la empresa company__JAOY9VHCI', 'Hola quiero completar una query. Tengo el supportCategory_1497612201762302662 y la empresa company__8VD90FMG4', 'Hola quiero completar una query. Tengo el supportCategory_1497644851762302663 y la empresa company__9G1G3MV0P', 'Hola quiero completar una query. Tengo el supportCategory_1497634611762302663 y la empresa company__FWP37ZIFM', 'Hola quiero completar una query. Tengo el supportCategory_149762050541762302872 y la empresa company__44XJYGG3L', 'Hola quiero completar una query. Tengo el supportCategory_149762443731762302917 y la empresa company__HJLEZ1WY6', 'Hola quiero completar una query. Tengo el supportCategory_1497634611762302663 y la empresa company__QRPQNRU25', 'Hola quiero completar una query. Tengo el supportCategory_149768521762302662 y la empresa company__0GQ4QH8N2', 'Hola quiero completar una query. Tengo el supportCategory_149766361762302662 y la empresa company__Y1XHYEPUS', 'Hola quiero completar una query. Tengo el supportCategory_149767793181762303635 y la empresa company_149763071541762302992', 'Hola quiero completar una query. Tengo el supportCategory_1497611711762302662 y la empresa company__ZLMCEWBY6', 'Hola quiero completar una query. Tengo el supportCategory_149764571762302662 y la empresa company__Y1XHYEPUS', 'Hola quiero completar una query. Tengo el supportCategory_1497621981762302663 y la empresa company__TLL5K8PC5']



print("casos_de_prueba = [")

for frase in frases_generadas5:
    f = io.StringIO()
    with redirect_stdout(f):
        query, ambas = procesar_query(frase)
    
    # Imprimimos con el formato de tupla y sangría para que sea legible
    print("    (")
    print(f"        {repr(frase)},")
    print(f"        {repr(query)}")
    print("    ),")

print("]")


    
    

casos_de_prueba = [
    (
        'Hola quiero completar una query. Tengo el supportCategory_149769471762302662 y la empresa company__IJVARZ08A',
        ['company__IJVARZ08A', 'supportCategory_149769471762302662', 'typeIncident__2', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__266']
    ),
    (
        'Hola quiero completar una query. Tengo el supportCategory_1497623551762302663 y la empresa company__Y1XHYEPUS',
        ['company__Y1XHYEPUS', 'supportCategory_1497623551762302663', 'typeIncident__2', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__294']
    ),
    (
        'Hola quiero completar una query. Tengo el supportCategory_1497622131762302663 y la empresa company_149766632941762303467',
        ['company__QCTRKWQRI', 'supportCategory_1497622131762302663', 'typeIncident__2', 'incidentOrigin__2', 'supportGroup_1497691762302662', 'employee__266']
    ),
    (
        'Hola quiero completar una query. Tengo el supportCategory_14976563921176

In [ ]:
## --- BUCLE PRINCIPAL PARALELIZADO ---
#
#casos_df = []
#casos_nv = []
#casos_ambos = []
#casos_dos = []
#nada = []
#
## Los contadores globales ahora solo viven y se modifican en el hilo principal
#contadores = {"vecesdf": 0, "vecesnv": 0}
#
## Usamos ThreadPoolExecutor para paralelizar (puedes ajustar max_workers según tu CPU)
#with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
#    
#    # Enviamos todas las tareas a procesar
#    futuros = [executor.submit(procesar_query, frase) for frase in frases_generadas]
#    
#    # as_completed nos va entregando los resultados a medida que van terminando, sin importar el orden
#    for futuro in concurrent.futures.as_completed(futuros):
#        texto, query, ambas, sumo_df, sumo_nv = futuro.result()
#        
#        # 2. Actualizamos los contadores de forma segura en el hilo principal
#        if sumo_df:
#            contadores["vecesdf"] += 1
#        if sumo_nv:
#            contadores["vecesnv"] += 1
#            
#        # 3. Clasificamos en las listas
#        if ambas:
#            if len(casos_ambos) < 10:
#                casos_ambos.insert(0, texto)
#        elif sumo_df and sumo_nv:
#            if len(casos_dos) < 10:
#                casos_dos.insert(0, texto)
#        elif sumo_nv:
#            if len(casos_nv) < 10:
#                casos_nv.insert(0, texto)
#        elif sumo_df:
#            if len(casos_df) < 10:
#                casos_df.insert(0, texto)
#        else:
#            nada.insert(0, texto)
#
#        # 4. Condición de parada
#        if len(casos_df) == 10 and len(casos_nv) == 10 and len(casos_dos) == 10 and len(casos_ambos) == 10:
#            print("¡Se han recolectado 10 casos de cada tipo!")
#            
#            # Cancelamos los procesos restantes en cola para ahorrar recursos
#            for f in futuros:
#                f.cancel()
#            break

In [ ]:
print(casos_df)
print(casos_nv)
print(casos_ambos)
print(casos_dos)

In [ ]:
print(nada)